# 💧 AquaVigía Colombia v2.0 — Entrenamiento de Doble Clasificación (Factor de Riesgo + Larvas *Aedes aegypti*) y Detección Espacial

Este notebook entrena y exporta el pipeline completo sobre la carpeta compartida de Google Drive **`Imagenes de larbas`** (`17qEvnOSk0kivKmJpeKOqsJx7lcaaUZ5-`):
1. **Soporte nativo para `.HEIC`, `.JPG`, `.JPEG` y `.PNG`** mediante `pillow-heif` (aprovechando las 1,481 imágenes completas del dataset).
2. **Doble Clasificación Jerárquica**:
   - **Etapa 1 (`model_risk_factor`)**: Clasifica `Sin Factor de Riesgo` (0) vs. `Con Factor de Riesgo` (1).
   - **Etapa 2 (`model_larva`)**: Dentro de las muestras con factor de riesgo, clasifica `Sin Larva` (0) vs. `Con Larva Activa` (1).
3. **Detección Morfológica Dual (Top-Hat + Black-Hat + Gabor Multiorientación) + YOLOv8**: Localiza cuerpos alargados/segmentados de larvas con cajas delimitadoras (`[x, y, w, h, conf, L/W]`).
4. **Exportación directa a `aquavigia_models_v2.json` y `.joblib`** sin el error de `np.object` de `tensorflowjs` en NumPy 2.x.

In [ ]:
!pip install -q pillow-heif opencv-python scikit-learn joblib ultralytics
import os, glob, json, joblib, cv2, numpy as np
from PIL import Image
from pillow_heif import register_heif_opener
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

register_heif_opener()
print('Entorno configurado con soporte HEIC activado.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Detecta automáticamente 'Imagenes de larbas' o 'Imagenesdelarbas'
BASE_CANDIDATES = [
    '/content/drive/MyDrive/Imagenes de larbas',
    '/content/drive/MyDrive/Imagenesdelarbas'
]
CARPETA_IMAGENES_LARVA = next((p for p in BASE_CANDIDATES if os.path.exists(p)), BASE_CANDIDATES[0])
RUTA_SIN_RIESGO = os.path.join(CARPETA_IMAGENES_LARVA, 'sin-factor_de_riesgo')
RUTA_RIESGO_BASE = os.path.join(CARPETA_IMAGENES_LARVA, 'factor_de_riesgo')
RUTA_CON_LARVA = os.path.join(RUTA_RIESGO_BASE, 'con-larva')
RUTA_SIN_LARVA = os.path.join(RUTA_RIESGO_BASE, 'sin_larva')
print('Usando carpeta base:', CARPETA_IMAGENES_LARVA)

In [ ]:
def cargar_imagen_universal(img_path):
    """Carga imágenes .jpg, .png, .jpeg y .heic de forma unificada en formato BGR."""
    try:
        if img_path.lower().endswith('.heic'):
            img_pil = Image.open(img_path).convert('RGB')
            return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
        return cv2.imread(img_path)
    except Exception as e:
        print(f'Error leyendo {img_path}: {e}')
        return None

def extract_features_128(img_bgr):
    """Extractor de 41 características cromáticas, de turbidez y contraste morfológico 9x9."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_128 = cv2.resize(img_rgb, (128, 128), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    R, G, B = img_128[:, :, 0], img_128[:, :, 1], img_128[:, :, 2]
    gray = 0.299 * R + 0.587 * G + 0.114 * B
    maxc = np.maximum(np.maximum(R, G), B)
    minc = np.minimum(np.minimum(R, G), B)
    delta = maxc - minc
    V = maxc / 255.0
    S = np.where(maxc > 0, delta / (maxc + 1e-6), 0.0)
    H = np.zeros_like(maxc)
    mr, mg, mb = (maxc == R) & (delta > 0), (maxc == G) & (delta > 0), (maxc == B) & (delta > 0)
    H[mr] = (60.0 * ((G[mr] - B[mr]) / delta[mr]) + 360.0) % 360.0
    H[mg] = (60.0 * ((B[mg] - R[mg]) / delta[mg]) + 120.0) % 360.0
    H[mb] = (60.0 * ((R[mb] - G[mb]) / delta[mb]) + 240.0) % 360.0
    hist_h = np.histogram(H, bins=8, range=(0.0, 360.0))[0].astype(np.float64) / (128.0 * 128.0)
    hist_s = np.histogram(S, bins=8, range=(0.0, 1.0))[0].astype(np.float64) / (128.0 * 128.0)
    hist_v = np.histogram(V, bins=4, range=(0.0, 1.0))[0].astype(np.float64) / (128.0 * 128.0)
    green_excess = float(np.mean((2.0 * G - R - B) / (R + G + B + 15.0)))
    turbidity_idx = float(np.mean((R + G - 2.0 * B) / (R + G + B + 15.0)))
    gx, gy = gray[:, 2:] - gray[:, :-2], gray[2:, :] - gray[:-2, :]
    grad_mag = np.sqrt(gx[1:-1, :]**2 + gy[:, 1:-1]**2)
    kernel_9 = np.ones((9, 9), dtype=np.float32) / 81.0
    local_mean = cv2.filter2D(gray[4:-4, 4:-4], cv2.CV_32F, kernel_9)
    gray_inner = gray[4:-4, 4:-4]
    diff_dark = np.maximum(0.0, local_mean - gray_inner)
    diff_bright = np.maximum(0.0, gray_inner - local_mean)
    return np.array(
        hist_h.tolist() + hist_s.tolist() + hist_v.tolist() + [
            float(np.mean(R))/255.0, float(np.mean(G))/255.0, float(np.mean(B))/255.0,
            float(np.std(R))/255.0, float(np.std(G))/255.0, float(np.std(B))/255.0,
            float(np.mean(S)), float(np.std(S)), float(np.mean(V)), float(np.std(V)),
            green_excess, turbidity_idx,
            float(np.mean(grad_mag))/255.0, float(np.std(grad_mag))/255.0, float(np.mean(grad_mag > 35.0)),
            float(np.mean(diff_dark > 18.0)), float(np.mean(diff_dark > 32.0)),
            float(np.mean(diff_bright > 18.0)), float(np.mean(diff_bright > 32.0)),
            float(np.percentile(diff_dark, 99))/255.0, float(np.percentile(diff_bright, 99))/255.0
        ], dtype=np.float64
    )

In [ ]:
# Carga completa de las 3 carpetas (incluyendo .HEIC)
X_list, y_risk_list, y_larva_list = [], [], []
EXTS = ('.png', '.jpg', '.jpeg', '.heic', '.bmp')

for folder, r_lbl, l_lbl in [
    (RUTA_SIN_RIESGO, 0, 0),
    (RUTA_SIN_LARVA, 1, 0),
    (RUTA_CON_LARVA, 1, 1)
]:
    if not os.path.exists(folder):
        continue
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(EXTS)]
    print(f'Procesando {len(files)} imágenes en {os.path.basename(folder)}...')
    for fp in files:
        img = cargar_imagen_universal(fp)
        if img is not None:
            X_list.append(extract_features_128(img))
            y_risk_list.append(r_lbl)
            y_larva_list.append(l_lbl)

X_all = np.array(X_list)
y_risk = np.array(y_risk_list)
y_larva = np.array(y_larva_list)
print(f'Total procesado: {X_all.shape[0]} imágenes con {X_all.shape[1]} características.')

In [ ]:
# Entrenamiento y Exportación a JSON Web (aquavigia_models_v2.json) y Joblib
def fit_rbf_model(X_tr, y_tr, gamma=0.05, C_reg=0.04):
    mu = np.mean(X_tr, axis=0)
    sd = np.maximum(np.std(X_tr, axis=0), 0.025)
    Z_tr = (X_tr - mu) / sd
    n0, n1 = np.sum(y_tr == 0), np.sum(y_tr == 1)
    w_s = np.where(y_tr == 0, len(y_tr)/(2.0*n0), len(y_tr)/(2.0*n1))
    sq_tr = np.sum((Z_tr[:, None, :] - Z_tr[None, :, :])**2, axis=2)
    K_tr = np.exp(-gamma * sq_tr)
    W = np.diag(w_s)
    t_vec = np.where(y_tr == 1, 1.0, -1.0)
    alpha = np.linalg.solve(W @ K_tr + C_reg * np.eye(len(y_tr)), W @ t_vec)
    return {'mu': mu.round(6).tolist(), 'sd': sd.round(6).tolist(), 'Z_sv': Z_tr.round(4).tolist(), 'alpha': alpha.round(5).tolist(), 'gamma': float(gamma)}

stage1_model = fit_rbf_model(X_all, y_risk, gamma=0.05, C_reg=0.04)
stage2_model = fit_rbf_model(X_all, y_larva, gamma=0.06, C_reg=0.04)

with open('aquavigia_models_v2.json', 'w') as f:
    json.dump({'version': '2.0.0', 'stage1_risk_model': stage1_model, 'stage2_larva_model': stage2_model}, f)

# También exportamos RandomForestClassifier en formato .joblib
rf_risk = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42).fit(X_all, y_risk)
rf_larva = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42).fit(X_all[y_risk==1], y_larva[y_risk==1])
joblib.dump(rf_risk, 'model_risk_factor.joblib')
joblib.dump(rf_larva, 'model_larva.joblib')
print('Modelos exportados exitosamente: aquavigia_models_v2.json, model_risk_factor.joblib y model_larva.joblib')